# Load thư viện + dataset

In [ ]:
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report


In [ ]:
df = pd.read_csv("mbti_1.csv")
df.head()


,type,posts
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...
1,ENTP,'I'm finding the lack of me in these posts ver...
2,INTP,'Good one _____ https://www.youtube.com/wat...
3,INTJ,"'Dear INTP, I enjoyed our conversation the o..."
4,ENTJ,'You're fired.|||That's another silly misconce...


# Preprocess

In [ ]:
# 1. Hàm làm sạch văn bản
def clean_text(text):
    """
    Hàm xử lý văn bản:
    - Chuyển đổi về chữ thường.
    - Loại bỏ URL và liên kết.
    - Loại bỏ các từ khóa MBTI để ngăn chặn rò rỉ dữ liệu (Data Leakage).
    - Loại bỏ các ký tự đặc biệt.
    """
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text) # Loại bỏ URL

    # Danh sách 16 từ khóa MBTI cần loại bỏ
    mbti_types = ['infj', 'entp', 'intp', 'intj', 'entj', 'enfj', 'infp', 'enfp',
                  'isfp', 'istp', 'isfj', 'istj', 'estp', 'esfp', 'estj', 'esfj']
    pattern = r'\b(?:' + '|'.join(mbti_types) + r')\b'
    text = re.sub(pattern, '', text)

    text = re.sub(r'[^a-z\s]', ' ', text) # Chỉ giữ lại chữ cái
    text = re.sub(r'\s+', ' ', text).strip() # Chuẩn hóa khoảng trắng
    return text

print("[THÔNG BÁO] Đang làm sạch dữ liệu và loại bỏ từ khóa MBTI...")
df['clean_posts'] = df['posts'].apply(clean_text)

# 2. Mã hóa nhãn thành 4 trục nhị phân
print("[THÔNG BÁO] Đang mã hóa nhãn thành 4 trục nhị phân (IE, NS, TF, JP)...")

def get_binary_labels(mbti_type):
    return pd.Series({
        'IE': 1 if 'I' in mbti_type else 0, # Hướng nội (1) vs Hướng ngoại (0)
        'NS': 1 if 'N' in mbti_type else 0, # Trực giác (1) vs Giác quan (0)
        'TF': 1 if 'T' in mbti_type else 0, # Lý trí (1) vs Cảm xúc (0)
        'JP': 1 if 'J' in mbti_type else 0  # Nguyên tắc (1) vs Linh hoạt (0)
    })

binary_targets = df['type'].apply(get_binary_labels)
df = pd.concat([df, binary_targets], axis=1)

print("[THÀNH CÔNG] Tiền xử lý dữ liệu hoàn tất.")

[THÔNG BÁO] Đang làm sạch dữ liệu và loại bỏ từ khóa MBTI...
[THÔNG BÁO] Đang mã hóa nhãn thành 4 trục nhị phân (IE, NS, TF, JP)...
[THÀNH CÔNG] Tiền xử lý dữ liệu hoàn tất.


# Train

In [ ]:
X = df['clean_posts']

traits = ['IE', 'NS', 'TF', 'JP']

for trait in traits:
    print(f"\n========== TRAINING FOR TRAIT: {trait} ==========")

    y = df[trait]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=100_000,
            min_df=5
        )),
        ("clf", LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            solver='liblinear'
        ))
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))



========== TRAINING FOR TRAIT: IE ==========
Accuracy: 0.7711815561959654
F1 Score: 0.8501321253303133
              precision    recall  f1-score   support

           0       0.50      0.53      0.52       400
           1       0.86      0.84      0.85      1335

    accuracy                           0.77      1735
   macro avg       0.68      0.69      0.68      1735
weighted avg       0.78      0.77      0.77      1735


========== TRAINING FOR TRAIT: NS ==========
Accuracy: 0.8576368876080691
F1 Score: 0.9171418986917141
              precision    recall  f1-score   support

           0       0.48      0.51      0.49       239
           1       0.92      0.91      0.92      1496

    accuracy                           0.86      1735
   macro avg       0.70      0.71      0.71      1735
weighted avg       0.86      0.86      0.86      1735


========== TRAINING FOR TRAIT: TF ==========
Accuracy: 0.7959654178674351
F1 Score: 0.7828220858895706
              precision    recall 